# DSCI 100 Project
## Topic: How do economic and structural influences affect the total cryptocurrency adoption across countries


#### Introduction: 
As traditional financial and economic systems face increasing strain in many parts of the world, cryptocurrency has emerged as a potential alternative financial tool and has been adopted by many different countries. This project examines whether broader forms of economic vulnerability, such as financial exclusion, income levels, macroeconomic instability and structural differences, including corruption index of other countries, are associated with higher levels of cryptocurrency adoption across countries.

The data is drawn from the World Bank, Corruption Index, Currency Depreciation data from Kaggle and Chainalysis, covering mainly the period from 2022 to 2025.

<p align="center">
  <img src="Crypto 2.jpg" width="600">
</p>

*Figure 1: Cryptocurrency as an emerging alternative to traditional financial systems.*

## Part 1: Baseline Model

Inflation reflects the degree of monetary instability within an economy, as rising prices reduce the purchasing power of a country’s currency. In such environments, individuals may be more likely to seek alternative stores of value or financial systems, such as cryptocurrency. To examine this relationship, we analyze cross-country data on inflation and crypto adoption.


### a) Cleaning the data:

In [235]:
import pandas as pd
inflation= pd.read_csv("data/Inflation.csv",skiprows=3)
inflation["avg_inflation"] = inflation[["2022", "2023", "2024"]].mean(axis=1)
inflation = inflation[["Country Name", "avg_inflation"]].copy()
inflation["Country Name"] = inflation["Country Name"].str.strip()
inflation

,Country Name,avg_inflation
0,Aruba,NaN
1,Africa Eastern and Southern,7.590817
2,Afghanistan,0.822069
3,Africa Western and Central,5.592821
4,Angola,21.079962
...,...,...
261,Kosovo,6.048095
262,"Yemen, Rep.",NaN
263,South Africa,5.825423
264,Zambia,12.287787


In [236]:
crypto= pd.read_csv("data/crypto_adoption_2024.csv")
crypto = crypto.rename(columns={"country": "Country Name"})
max_rank = crypto["overall_rank_2024"].max()
crypto["adoption_score"] = 1 - (crypto["overall_rank_2024"] - 1) / (max_rank - 1)
crypto

,Country Name,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi,adoption_score
0,India,Central & Southern Asia and Oceania,1,1,1,3,2,1.000000
1,Nigeria,Sub-Saharan Africa,2,5,2,2,3,0.993333
2,Indonesia,Central & Southern Asia and Oceania,3,6,6,1,1,0.986667
3,United States,North America,4,2,12,4,4,0.980000
4,Vietnam,Central & Southern Asia and Oceania,5,3,3,6,5,0.973333
...,...,...,...,...,...,...,...,...
146,Mauritius,Sub-Saharan Africa,147,126,130,150,150,0.026667
147,Belize,Latin America,148,152,158,123,149,0.020000
148,Rep. of Congo,Sub-Saharan Africa,149,147,142,151,144,0.013333
149,Mali,Sub-Saharan Africa,150,88,122,156,152,0.006667


In [237]:
inflation_crypto = inflation.merge(crypto, on="Country Name", how="inner")
inflation_crypto.sort_values(by="overall_rank_2024")

,Country Name,avg_inflation,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi,adoption_score
54,India,5.767071,Central & Southern Asia and Oceania,1,1,1,3,2,1.000000
89,Nigeria,25.582945,Sub-Saharan Africa,2,5,2,2,3,0.993333
53,Indonesia,3.353455,Central & Southern Asia and Oceania,3,6,6,1,1,0.986667
125,United States,5.022888,North America,4,2,12,4,4,0.980000
123,Ukraine,13.178215,Eastern Europe,6,7,5,5,6,0.966667
...,...,...,...,...,...,...,...,...,...
0,Aruba,NaN,Latin America,146,154,157,138,138,0.033333
85,Mauritius,7.143752,Sub-Saharan Africa,147,126,130,150,150,0.026667
18,Belize,4.652098,Latin America,148,152,158,123,149,0.020000
78,Mali,4.962226,Sub-Saharan Africa,150,88,122,156,152,0.006667


In [238]:
bank=pd.read_csv("data/Bank access.csv",skiprows=3)
bank["bank_access"] = bank["2024"]
bank_clean = bank.dropna(subset=["bank_access"])
bank_clean = bank_clean[["Country Name", "bank_access"]]
bank_clean

,Country Name,bank_access
5,Albania,46.069251
9,Argentina,81.744245
10,Armenia,71.373473
13,Australia,98.010378
14,Austria,99.508497
...,...,...
259,World,78.742404
261,Kosovo,64.161772
263,South Africa,81.125966
264,Zambia,72.702425


In [239]:
GDP= pd.read_csv("data/GDP.csv", skiprows=4)
GDP["avg_gdp"] = GDP[["2022", "2023", "2024"]].mean(axis=1)
GDP=GDP[["Country Name","2022", "2023", "2024","avg_gdp"]]
GDP

,Country Name,2022,2023,2024,avg_gdp
0,Aruba,30975.998912,35718.753119,39498.594129,35397.782054
1,Africa Eastern and Southern,1679.327622,1571.449189,1615.396356,1622.057722
2,Afghanistan,357.261153,413.757895,NaN,385.509524
3,Africa Western and Central,2138.473153,1841.855064,1411.337029,1797.221749
4,Angola,3682.113151,2916.136633,2665.874448,3088.041411
...,...,...,...,...,...
261,Kosovo,5290.950065,6221.206188,7023.065985,6178.407413
262,"Yemen, Rep.",NaN,NaN,NaN,NaN
263,South Africa,6534.248678,6034.272090,6267.186814,6278.569194
264,Zambia,1447.123101,1330.727806,1187.109434,1321.653447


In [240]:
inflation_final = inflation[["Country Name", "avg_inflation"]].copy()
inflation_final["Country Name"] = inflation_final["Country Name"].str.strip()

crypto["Country Name"] = crypto["Country Name"].str.strip()

bank = bank[["Country Name", "bank_access"]].copy()
bank["Country Name"] = bank["Country Name"].str.strip()


GDP = GDP[["Country Name", "avg_gdp"]].copy()
GDP["Country Name"] = GDP["Country Name"].str.strip()


merged_all = inflation_final.merge(GDP, on="Country Name", how="inner") \
                            .merge(crypto, on="Country Name", how="inner") \
                            .merge(bank, on="Country Name", how="inner")


### b) Regression Result:

In [241]:
baseline_data = merged_all[[
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access"
]].dropna()

from sklearn.linear_model import LinearRegression

X = baseline_data[["avg_inflation", "avg_gdp", "bank_access"]]
y = baseline_data["adoption_score"]

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)
print("R^2:", model.score(X, y))

Intercept: 0.2646720002112302
Coefficients: [ 1.74956486e-03 -1.66124966e-06  3.80155634e-03]
R^2: 0.06340298076872386


In [242]:
import pandas as pd
import altair as alt

# Create dataframe with actual and predicted values
baseline_plot_df = pd.DataFrame({
    "actual": y,
    "predicted": model.predict(X)
})

# Scatter plot
points = alt.Chart(baseline_plot_df).mark_circle(opacity=0.5).encode(
    x=alt.X("actual", title="Actual Crypto Adoption Score"),
    y=alt.Y("predicted", title="Predicted Crypto Adoption Score")
).properties(
    title="Baseline Model: Actual vs Predicted Crypto Adoption"
)

baseline_plot

alt.Chart(...)

### Results & Findings: Baseline Model

The baseline regression results indicate that inflation, GDP, and bank access have limited explanatory power in predicting cryptocurrency adoption across countries, with an R² of approximately 0.063.

Inflation shows a small positive relationship with cryptocurrency adoption, suggesting that higher inflation may slightly increase the demand for alternative financial assets. GDP, however, exhibits a near-zero effect, indicating that overall income levels do not play a significant role in explaining cross-country differences in adoption.

Interestingly, bank access is positively associated with cryptocurrency adoption and has the largest coefficient among the variables. This suggests that cryptocurrency is not solely a substitute for traditional financial systems, but may also be adopted within more developed financial environments.

Despite these relationships, the overall explanatory power of the model remains weak. This indicates that traditional macroeconomic variables alone are insufficient to account for variations in cryptocurrency adoption.

Overall, the findings suggest that cryptocurrency adoption is likely influenced by factors beyond standard economic indicators, pointing to the potential importance of structural and institutional conditions.


## Part 2: Regional Differences in Crypto Adoption
Given the weak relationships observed in the regression analysis, it is necessary to consider alternative perspectives. In particular, examining regional differences may reveal patterns that are not captured by individual economic indicators.


In [243]:
model_data2 = merged_all[[
    "Country Name",
    "region",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "adoption_score"
]].dropna()

model_data2.head()

,Country Name,region,avg_inflation,avg_gdp,bank_access,adoption_score
3,Albania,Central & Western Europe,4.566474,9621.868950,46.069251,0.280000
5,Argentina,Latin America,141.934541,14064.606545,81.744245,0.906667
6,Armenia,Middle East & North Africa,3.630281,7762.425245,71.373473,0.493333
7,Australia,Central & Southern Asia and Oceania,5.117575,64943.960686,98.010378,0.746667
8,Austria,Central & Western Europe,6.432973,55728.385154,99.508497,0.513333


In [244]:
region_avg = model_data2.groupby("region")["adoption_score"].mean().reset_index()

alt.Chart(region_avg).mark_bar().encode(
    y=alt.Y("region", sort='-x', title="Region"),
    x=alt.X("adoption_score", title="Adoption Score"),
    color="region"
).properties(title="Average Crypto Adoption by Region")

alt.Chart(...)

#### Result & Findings: 
The bar chart shows clear variation in average cryptocurrency adoption across regions. North America has the highest adoption score, followed by Central & Southern Asia and Eastern Europe, while Sub-Saharan Africa and Latin America have noticeably lower levels.
The regional analysis reveals clear differences in cryptocurrency adoption across countries. This suggests that cryptocurrency adoption is influenced more by broader regional conditions than by individual economic variables alone, highlighting the importance of contextual and structural factors.

## Part 3: Similarity among High-Adoption Countries
Finally, to explore whether high cryptocurrency adoption is associated with a common economic pattern, this section focuses on countries with the highest adoption levels. By comparing their economic characteristics—such as inflation, GDP, and financial access with the overall sample, we can assess whether these countries share similar underlying conditions.


In [245]:
import pandas as pd

h4_data = merged_all[[
    "Country Name",
    "region",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "adoption_score"
]].dropna()

threshold = h4_data["adoption_score"].quantile(0.75)
high_adoption = h4_data[h4_data["adoption_score"] >= threshold]
high_adoption.head()

,Country Name,region,avg_inflation,avg_gdp,bank_access,adoption_score
5,Argentina,Latin America,141.934541,14064.606545,81.744245,0.906667
13,Bangladesh,Central & Southern Asia and Oceania,9.348735,2620.306594,43.283973,0.773333
20,Brazil,Latin America,6.080378,9989.823824,86.381178,0.940000
22,Canada,North America,4.354462,54939.158984,98.404713,0.886667
25,China,Eastern Asia,0.808847,13074.977345,89.383032,0.873333


In [246]:
high_adoption[["avg_inflation", "avg_gdp", "bank_access"]].mean()

avg_inflation       12.484310
avg_gdp          18682.601698
bank_access         75.406054
dtype: float64

In [247]:
h4_data[["avg_inflation", "avg_gdp", "bank_access"]].mean()

avg_inflation        9.195744
avg_gdp          20291.901503
bank_access         72.445462
dtype: float64

In [248]:
high_adoption["region"].value_counts()

Central & Southern Asia and Oceania    8
Central & Western Europe               5
Sub-Saharan Africa                     5
Latin America                          4
North America                          2
Eastern Asia                           2
Eastern Europe                         2
Middle East & North Africa             1
Name: region, dtype: int64

In [217]:
import pandas as pd
import altair as alt

compare_df = pd.DataFrame({"group": ["High Adoption", "All Countries"],
    "avg_inflation": [high_adoption["avg_inflation"].mean(),
        h4_data["avg_inflation"].mean()],
    
        "avg_gdp": [
        high_adoption["avg_gdp"].mean(),
        h4_data["avg_gdp"].mean()],
                           
    "bank_access": [
        high_adoption["bank_access"].mean(),
        h4_data["bank_access"].mean()]})

compare_melt = compare_df.melt(
    id_vars="group",
    var_name="variable",
    value_name="value")

alt.Chart(compare_melt).mark_bar().encode(
    y=alt.Y("group", title="Group"),
    x=alt.X("value", title="Average Value"),
    color=alt.Color("group", title="Group")
).facet(row=alt.Row("variable", title=None)
).resolve_scale(x="independent"
).properties(
    title="High-Adoption Countries vs All Countries")

alt.FacetChart(...)

#### Result/Findings:
The faceted charts compare key economic indicators between high-adoption countries and the overall sample. High-adoption countries exhibit slightly higher average inflation and bank access, while GDP levels are marginally lower. However, the differences across all variables are relatively small, indicating that high-adoption countries do not share a single, distinct economic profile.


## Part 4: Extended Model – Currency Instability and Institutional Factors
To address the limitations of the baseline model, this section introduces additional variables that better reflect real-world drivers of cryptocurrency adoption. In particular, currency depreciation and corruption are included to capture monetary instability and institutional trust.

Currency depreciation reflects the loss of value in local currencies, which may encourage individuals to adopt cryptocurrencies as alternative stores of value. The corruption index is used as a proxy for institutional trust, where lower trust in government and financial systems may increase reliance on decentralized financial tools.

In [222]:
import pandas as pd
currency_clean = pd.read_csv("data/currency_clean.csv")
corruption_index=pd.read_csv("data/index.csv")
corruption_index

,CPI Rank,Country,Country Code,Region,Corruption Perceptions Index (CPI),Standard Error,Lower Confidence Interval,Upper Confidence Interval,World Bank CPIA,World Economic Forum EOS,...,African Development Bank CPIA,IMD World Competitiveness Yearbook,Bertelsmann Foundation Sustainable Governance Index,World Justice Project Rule of Law Index,PRS International Country Risk Guide,Varities of Democracy Project,Economist Intelligence Unit Country Ratings,Freedom House Nations in Transit Ratings,PERC Asia Risk Guide,Sources
0,1,New Zealand,NZL,Asia Pacific,90,2.56,86,94,NaN,90.0,...,NaN,95.0,99.0,79.0,93.0,NaN,90.0,NaN,NaN,7
1,1,Denmark,DNK,Europe and Central Asia,90,2.46,86,94,NaN,85.0,...,NaN,98.0,99.0,85.0,93.0,NaN,90.0,NaN,NaN,7
2,3,Finland,FIN,Europe and Central Asia,89,1.46,87,92,NaN,91.0,...,NaN,94.0,90.0,85.0,93.0,NaN,90.0,NaN,NaN,7
3,4,Sweden,SWE,Europe and Central Asia,88,1.33,85,90,NaN,86.0,...,NaN,86.0,90.0,85.0,93.0,NaN,90.0,NaN,NaN,7
4,5,Switzerland,CHE,Europe and Central Asia,86,1.57,83,89,NaN,80.0,...,NaN,88.0,90.0,NaN,85.0,NaN,90.0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,170,Sudan,SDN,Sub-Saharan Africa,14,2.99,9,19,2.0,NaN,...,11.0,NaN,NaN,NaN,6.0,22.0,19.0,NaN,NaN,7
172,173,Syria,SYR,Middle East and North Africa,13,1.97,10,16,NaN,NaN,...,NaN,NaN,NaN,NaN,15.0,12.0,19.0,NaN,NaN,5
173,174,Korea (North),PRK,Asia Pacific,12,1.39,10,15,NaN,NaN,...,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,3
174,175,South Sudan,SSD,Sub-Saharan Africa,11,3.21,5,16,2.0,NaN,...,5.0,NaN,NaN,NaN,NaN,19.0,NaN,NaN,NaN,5


In [228]:
extended_data = merged_all.merge(
    currency_clean,
    on="Country Name",
    how="left")
extended_data

,Country Name,avg_inflation,avg_gdp,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi,adoption_score,bank_access,currency_dep
0,Aruba,NaN,35397.782054,Latin America,146,154,157,138,138,0.033333,NaN,0.002688
1,Afghanistan,0.822069,385.509524,Central & Southern Asia and Oceania,31,40,56,8,46,0.800000,NaN,0.373404
2,Angola,21.079962,3088.041411,Sub-Saharan Africa,139,114,110,149,145,0.080000,NaN,NaN
3,Albania,4.566474,9621.868950,Central & Western Europe,109,110,117,111,100,0.280000,46.069251,-0.192556
4,Andorra,NaN,46176.707752,Central & Western Europe,111,125,133,103,79,0.266667,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
125,United States,5.022888,80741.183929,North America,4,2,12,4,4,0.980000,97.023098,NaN
126,Uzbekistan,10.343830,2873.111931,Central & Southern Asia and Oceania,33,46,40,30,26,0.786667,59.658383,NaN
127,South Africa,5.825423,6278.569194,Sub-Saharan Africa,30,19,28,40,35,0.806667,81.125966,1.108909
128,Zambia,12.287787,1321.653447,Sub-Saharan Africa,58,35,54,80,78,0.620000,72.702425,NaN


In [229]:
corruption = corruption_index.rename(columns={
    "Country": "Country Name",
    "Corruption Perceptions Index (CPI)": "corruption_index"
})

extended_data = merged_all.merge(
    currency_clean,
    on="Country Name",
    how="left"
).merge(corruption,on="Country Name",how="left")


In [230]:
from sklearn.linear_model import LinearRegression
model_data_ext = extended_data[[
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]].dropna()

X_ext = model_data_ext[[
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]]

y_ext = model_data_ext["adoption_score"]

model_ext = LinearRegression()
model_ext.fit(X_ext, y_ext)

print("Intercept:", model_ext.intercept_)
print("Coefficients:", model_ext.coef_)
print("R^2:", model_ext.score(X_ext, y_ext))

Intercept: 0.51123576826791
Coefficients: [ 6.29756655e-03 -2.73194177e-06  3.40492085e-03 -2.41314608e-03
 -2.81413094e-03]
R^2: 0.1939708226821042


In [231]:
model_data_ext = extended_data[[
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]].dropna()

plot_df = model_data_ext.copy()
x_min, x_max = -1, 10

currency_grid = pd.DataFrame({"currency_dep": np.linspace(x_min, x_max, 100)})

currency_grid["avg_inflation"] = plot_df["avg_inflation"].mean()
currency_grid["avg_gdp"] = plot_df["avg_gdp"].mean()
currency_grid["bank_access"] = plot_df["bank_access"].mean()
currency_grid["corruption_index"] = plot_df["corruption_index"].mean()

currency_grid["predicted"] = model_ext.predict(currency_grid[[
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"]])

plot_df_trim = plot_df[
    (plot_df["currency_dep"] >= x_min) &
    (plot_df["currency_dep"] <= x_max)]

base_plot = alt.Chart(plot_df_trim).mark_circle(opacity=0.4).encode(
    x=alt.X("currency_dep", title="Currency Depreciation"),
    y=alt.Y("adoption_score", title="Crypto Adoption Score")
).properties(
    title="Predicted Crypto Adoption by Currency Depreciation"
)

currency_plot

alt.LayerChart(...)

In [232]:
from sklearn.linear_model import LinearRegression
model_data_ext = extended_data[[
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"]].dropna()

X_ext = model_data_ext[[
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]]

y_ext = model_data_ext["adoption_score"]

model_ext = LinearRegression()
model_ext.fit(X_ext, y_ext)

print("Intercept:", model_ext.intercept_)
print("Coefficients:", model_ext.coef_)
print("R^2:", model_ext.score(X_ext, y_ext))

Intercept: 0.51123576826791
Coefficients: [ 6.29756655e-03 -2.73194177e-06  3.40492085e-03 -2.41314608e-03
 -2.81413094e-03]
R^2: 0.1939708226821042


In [234]:
import pandas as pd
import numpy as np
import altair as alt

plot_df = model_data_ext.copy()

# Create grid for corruption_index
corruption_grid = pd.DataFrame({
    "corruption_index": np.linspace(
        plot_df["corruption_index"].min(),
        plot_df["corruption_index"].max(),
        100
    )
})

# Hold other variables constant at their means
corruption_grid["avg_inflation"] = plot_df["avg_inflation"].mean()
corruption_grid["avg_gdp"] = plot_df["avg_gdp"].mean()
corruption_grid["bank_access"] = plot_df["bank_access"].mean()
corruption_grid["currency_dep"] = plot_df["currency_dep"].mean()

corruption_grid["predicted"] = model_ext.predict(corruption_grid[[
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]])

base_plot_corruption = alt.Chart(plot_df).mark_circle(opacity=0.4).encode(
    x=alt.X("corruption_index", title="CPI Score (Higher = Less Corruption)"),
    y=alt.Y("adoption_score", title="Crypto Adoption Score")
)

corruption_plot = base_plot_corruption + alt.Chart(
    corruption_grid,
    title="Predicted Crypto Adoption by CPI Score"
).mark_line(color="#ff7f0e").encode(
    x="corruption_index",
    y="predicted"
)

corruption_plot

alt.LayerChart(...)

To improve the model, currency depreciation and institutional quality were added. The extended model increases R² from 0.06 to approximately 0.19, indicating a better explanation of cryptocurrency adoption.

Inflation remains positively associated with adoption, while GDP shows little effect. Bank access is also positive, suggesting that cryptocurrency may complement existing financial systems.

Currency depreciation has a negative relationship with adoption, indicating that extreme instability may limit participation due to structural constraints. The corruption index is also negatively related to adoption; since higher CPI scores indicate less corruption, this suggests that countries with higher corruption tend to have higher cryptocurrency adoption, possibly reflecting lower trust in formal institutions.

Overall, cryptocurrency adoption is not driven solely by economic conditions, but by a combination of economic pressure and institutional context.



## Conclusion

The results show that traditional macroeconomic variables alone are insufficient to explain cryptocurrency adoption. Instead, adoption appears to occur where both incentive and capacity exist. These findings highlight the importance of considering both economic and institutional factors in understanding global cryptocurrency adoption.

<p align="center">
  <img src="crypto_crash.jpeg" width="600">
</p>

*Figure 2: Cryptocurrency remain unstable in the finanacial market*